# 23 — Paired-bootstrap gate fix + today's three candidates (2026-09-11)

An Opus deep review (requested after the user felt "stuck" post-submission-2)
found two things before endorsing any new work:

1. **No bug in the shipped recipe.** `submission_src/main.py` was audited
   line-by-line against `notebooks/22_calibration_refit_rowwise_cv.ipynb`'s
   adopted recipe (6-variant/150-checkpoint ensemble, logit-space pooling,
   `a=0.8558, b=0.5296, c=-0.0905`) — they match exactly. The CV-to-leaderboard
   gap (0.3617 -> 0.4185) is real distribution shift plus CV selection
   optimism, not a wiring bug: submission 2's real leaderboard log loss is
   within **+0.0020** of the theoretical optimum for a model with its own
   real-test AUROC (0.8891) -- calibration is essentially exhausted, and any
   further gain has to come from *discrimination* (AUROC), which is much
   harder to move cheaply.
2. **Notebook 22's own decision rule (`op06denoise`) used the wrong
   denominator**: it compared a *paired* delta (candidate vs. current, same
   rows) against the **unpaired across-fold sd of absolute scores** (~0.025,
   driven mostly by which rows landed in which fold -- identical for both
   arms of the comparison). That denominator can't resolve effects smaller
   than its own ~0.011 SEM, which is exactly the size of everything left to
   test. This notebook fixes that: every comparison below uses
   `evaluate.paired_bootstrap_ci` (bootstrap the *rows*, compute the log-loss
   delta on the same resampled rows for both arms) -- already implemented in
   `src/evaluate.py`, just never used for this kind of comparison before.
   **Decision rule for every cell below: adopt only if the whole 95% CI is
   negative AND `|delta| > 0.003`** (below ~0.003 the ~92% CV-to-LB transfer
   ratio measured across submissions 1 and 2 would leave nothing visible on
   the real leaderboard anyway).

Three genuinely new candidates were surfaced by a literature/forum research
pass earlier the same day, none tested against this project's actual data
before now:

- **Candidate 1 (Khachnaoui et al. 2023 -- `RESOURCES.md`, "bilinear fusion"
  paper)**: their literal method (outer-product fusion of two CNN backbones'
  feature maps) does not transfer here -- this project's 30 CNN members are
  independently initialized, so their penultimate embeddings live in
  mutually unaligned spaces with no shared basis, and no embedding is saved
  anywhere in this codebase (`grep -rniE "embedding|penultimate|forward_features"`
  across `src/`, `scripts/`, `submission_src/`, all notebooks: zero hits).
  What *is* cheap and viable is the scalar remnant of the idea: second-order
  interaction/quadratic terms between the two existing blend scores, plus
  the classical-baseline's own engineered features
  (`data/processed/baseline_features.csv`), which the current 2-feature
  blend (`logit(cnn_p)`, `logit(baseline_p)`) never sees directly.
- **Candidate 2 (Ithapu, Singh & Johnson 2015, "Randomized Deep Learning
  Methods for Clinical Trial Enrichment and Design in Alzheimer's Disease"
  -- Ch. 15 of *Deep Learning for Medical Image Analysis*, Zhou/Greenspan/
  Shen eds., `RESOURCES.md`)**: their core mechanism is that a regularized
  (ridge) linear combination of near-decorrelated weak learners reduces
  variance further than uniform averaging while staying unbiased -- their
  eq. (15.6)-(15.8). Applied here: instead of uniformly logit-pooling the 6
  CNN variants before the 2-feature blend, fit a regularized weighting
  across the 6 variant-level pooled logits + the baseline logit. This
  project's own ensemble-composition history (every gain has come from
  *adding* decorrelated members, never from reweighting the ones already
  present) predicts this will be close to null, but it's free to check.
- **Candidate 3 (Budd et al. 2023, EJNMMI, "Automated identification of
  uncertain cases in deep learning-based classification of dopamine
  transporter SPECT," DOI 10.1007/s00259-023-06566-w -- the most directly
  on-topic paper found today: real multi-site FP-CIT DAT-SPECT, not PPMI)**:
  their full method trains two additional asymmetric-loss ensembles and uses
  their disagreement to flag uncertain cases for human review -- not free
  (1-2 new training runs). The free precursor tested here is whether the
  disagreement *already present* across the 30 existing CNN members (their
  per-row logit standard deviation) carries information a log-loss-scored
  blend can use, before spending any GPU time on new asymmetric models. If
  this null result holds, candidate 3 is dead at zero additional cost --
  the full asymmetric-ensemble version targets a calibration seam already
  measured at +0.0020, so its ceiling is tiny regardless.

**Pre-registered expectation for all three (stated before running, per this
project's own discipline): null.** The diminishing-returns pattern across
every prior roadmap item (-0.0393 -> -0.0105 -> -0.0044 -> -0.0049 ->
-0.0016 rejected -> -0.0001 rejected) and the +0.0020 calibration-exhaustion
finding both point the same way. Running them anyway costs one CPU-only
notebook and a few minutes.

**Data handling**: loads real row-level labels, engineered baseline
features, and existing OOF prediction arrays (all already used by prior
`[RUN ME]` notebooks in this repo) -- per the AI-assistant data rule
(`README.md`), this is **[RUN ME]**: run it yourself, share back only the
printed aggregate numbers. CPU-only, no GPU, no volume cache -- well under a
minute total.

In [ ]:
# [RUN ME] -- loads real row-level labels, engineered baseline features, and
# existing OOF prediction arrays. CPU-only, no GPU, no volume cache --
# self-contained, does not assume any earlier notebook ran in this kernel.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

import config
import evaluate
import model

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
# baseline_features.csv also carries its own is_pathologic column -- select
# only the engineered feature columns so the merge doesn't collide and
# suffix it into is_pathologic_x/_y (notebook 22 hit the same shape of
# issue with inplane_family and already selects columns explicitly).
features_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family", "abs_asym", "striatal_ratio"]
]
labeled_df = labels_df.merge(features_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
y_true = labeled_df[config.TARGET_COLUMN].to_numpy()
family = labeled_df["inplane_family"].to_numpy()
abs_asym = labeled_df["abs_asym"].to_numpy(dtype=float)
striatal_ratio = labeled_df["striatal_ratio"].to_numpy(dtype=float)

repeat_seeds = list(range(config.SEED, config.SEED + 5))
VARIANT_PREFIXES = model.PRODUCTION_VARIANT_PREFIXES  # the 6 shipped variants (no denoise)

# 30 individually-OOF CNN "members" (6 variants x 5 seeds) -- same arrays
# notebook 22 used, each element is exactly one held-out checkpoint's
# prediction per row.
cnn_members = {
    (prefix, s): np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy")
    for prefix in VARIANT_PREFIXES for s in repeat_seeds
}
cnn_arrays = list(cnn_members.values())

baseline_oof_repeats = [np.load(config.DATA_PROCESSED / f"baseline_oof_seed{s}.npy") for s in repeat_seeds]
baseline_pooled = np.mean(baseline_oof_repeats, axis=0)

print(f"{len(cnn_arrays)} CNN members loaded (6 variants x 5 seeds), "
      f"{len(baseline_oof_repeats)} baseline repeats pooled, {len(labeled_df)} rows.")

In [ ]:
# [RUN ME] (no new data access -- defines helpers used by every cell below).
EPS = 1e-6


def to_logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def logit_mean(arrays):
    """Adopted pooling method (notebook 22, delta -0.0020 vs prob-space)."""
    return 1.0 / (1.0 + np.exp(-np.mean([to_logit(a) for a in arrays], axis=0)))


FOLDS = evaluate.make_folds(y_true, family, n_splits=config.N_FOLDS, random_state=config.RANDOM_STATE)


def oof_predict(X, y=y_true, folds=FOLDS, regularized=False, cs=np.logspace(-2, 2, 9), inner_cv=5):
    """Honest row-wise-CV out-of-fold prediction: for each outer fold, fit
    on the train rows and predict the held-out rows, then reassemble in
    original row order. Returns one array, same length/order as y.

    `regularized=True` uses `LogisticRegressionCV` (L2, `cs` grid, `inner_cv`
    folds tuned *inside* each outer fold's train rows only -- a proper
    nested design, never selects the penalty on rows it's then scored on).
    Use this for any candidate whose feature count/collinearity makes an
    unregularized fit risky (candidates 1 and 2 below); the shipped 2-feature
    blend and candidate 3's 4-feature blend use the plain (unregularized)
    fit, matching how the shipped recipe (notebook 22) was actually fit.
    """
    oof = np.empty(len(y), dtype=float)
    for train_idx, test_idx in folds:
        if regularized:
            # l1_ratios=(0.0,) is the current (non-deprecated) way to ask
            # for pure L2 (this sklearn build deprecated the `penalty=`
            # kwarg on LogisticRegressionCV in favor of l1_ratios/Cs); we
            # only ever call predict_proba on the fitted estimator, never
            # touch the legacy .scores_/.coefs_paths_ attribute shapes, so
            # opting into the new (non-legacy) attribute layout is safe.
            clf = LogisticRegressionCV(Cs=cs, cv=inner_cv, l1_ratios=(0.0,),
                                        max_iter=2000, scoring="neg_log_loss",
                                        use_legacy_attributes=False)
        else:
            clf = LogisticRegression(C=np.inf, max_iter=1000)
        clf.fit(X[train_idx], y[train_idx])
        oof[test_idx] = clf.predict_proba(X[test_idx])[:, 1]
    return oof


def blend_features(cnn_p, baseline_p):
    return np.column_stack([to_logit(cnn_p), to_logit(baseline_p)])


def paired_gate(label, candidate_oof, reference_oof, min_effect=0.003, seed=config.RANDOM_STATE):
    """The fixed gate: paired bootstrap over rows (evaluate.paired_bootstrap_ci),
    not delta-vs-unpaired-fold-sd (notebook 22's op06 bug). Adopt only if the
    whole 95% CI is negative (candidate reliably beats reference on every
    plausible resample) AND the point delta exceeds min_effect (below which
    the ~92% CV-to-LB transfer measured on submissions 1-2 would not produce
    a visible leaderboard change anyway).
    """
    score_candidate = evaluate.log_loss_score(y_true, candidate_oof)
    score_reference = evaluate.log_loss_score(y_true, reference_oof)
    delta = score_candidate - score_reference
    ci_low, ci_high = evaluate.paired_bootstrap_ci(y_true, candidate_oof, reference_oof, seed=seed)
    clears = (ci_high < 0) and (abs(delta) > min_effect)
    print(f"{label}")
    print(f"  candidate row-CV log loss = {score_candidate:.4f}   reference = {score_reference:.4f}")
    print(f"  delta (candidate - reference) = {delta:+.4f}   "
          f"95% paired-bootstrap CI = [{ci_low:+.4f}, {ci_high:+.4f}]")
    print(f"  -> {'CLEARS the gate (adopt)' if clears else 'does NOT clear the gate (keep reference)'} "
          f"(rule: whole CI < 0 AND |delta| > {min_effect})\n")
    return {"label": label, "score_candidate": score_candidate, "score_reference": score_reference,
            "delta": delta, "ci_low": ci_low, "ci_high": ci_high, "clears": clears}


print(f"{len(FOLDS)} row-wise folds built (evaluate.make_folds, stratified on target x family).")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# The shipped recipe's own honest per-row OOF prediction (6-variant,
# logit-pooled, 2-feature logistic blend -- notebook 22's adopted recipe),
# refit here as a full-length OOF array so it can serve as the reference
# arm for every paired-bootstrap comparison below. This should closely
# match notebook 22's reported row-wise CV mean (0.3617) -- if it doesn't,
# that's itself a finding worth flagging before trusting anything else here.
cnn_pooled = logit_mean(cnn_arrays)
current_oof = oof_predict(blend_features(cnn_pooled, baseline_pooled))

current_log_loss = evaluate.log_loss_score(y_true, current_oof)
current_auroc = evaluate.auroc_score(y_true, current_oof)
current_ece = evaluate.expected_calibration_error(y_true, current_oof)
print(f"shipped recipe, honest row-wise CV: log loss={current_log_loss:.4f}  "
      f"AUROC={current_auroc:.4f}  ECE={current_ece:.4f}")
print("for comparison -- notebook 22's reported figure: log loss=0.3617")
print("for comparison -- real leaderboard (submission 2): log loss=0.4185  AUROC=0.8891")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Opus review ask #1: re-check whether the k=30-fit CNN coefficient `a` is
# close to its k=150 (production ensemble size) extrapolation, rather than
# trusting notebook 22's own printed cell without re-deriving it here.
# Diagnostic only -- does not gate a decision on its own; the log-loss
# surface near the optimum is quadratically flat, so even a real gap here
# is expected to cost well under the 0.003 adoption threshold.
rng = np.random.RandomState(config.RANDOM_STATE)
K_VALUES = [1, 2, 3, 5, 6, 10, 15, 20, 25, 30]
R_DRAWS = 20

a_by_k = []
for k in K_VALUES:
    a_draws = []
    for _ in range(R_DRAWS):
        subset_idx = rng.choice(len(cnn_arrays), size=k, replace=False)
        subset = [cnn_arrays[i] for i in subset_idx]
        pooled_k = logit_mean(subset)
        clf = LogisticRegression(C=np.inf, max_iter=1000)
        clf.fit(blend_features(pooled_k, baseline_pooled), y_true)
        a_draws.append(clf.coef_[0][0])
    a_by_k.append(float(np.mean(a_draws)))

inv_k = 1.0 / np.array(K_VALUES, dtype=float)
slope, intercept_a = np.polyfit(inv_k, a_by_k, 1)
a_extrapolated_150 = intercept_a + slope / 150

clf_shipped = LogisticRegression(C=np.inf, max_iter=1000)
clf_shipped.fit(blend_features(cnn_pooled, baseline_pooled), y_true)
a_shipped = clf_shipped.coef_[0][0]

print(f"a by k: {dict(zip(K_VALUES, [round(v, 4) for v in a_by_k]))}")
print(f"linear fit in 1/k: a ~= {intercept_a:.4f} + {slope:.4f}/k")
print(f"extrapolated a at k=150: {a_extrapolated_150:.4f}   shipped a (k=30 fit, full data): {a_shipped:.4f}")
print(f"difference: {a_extrapolated_150 - a_shipped:+.4f}  "
      "(small -> current a is a safe stand-in for k=150; large -> worth a re-fit)")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Opus review ask #2: PROB_CLIP=(0.005, 0.995) is applied in production
# AFTER the blend, but was never itself part of the row-wise CV that
# produced 0.3617 -- measure its effect directly (deterministic transform
# of the same predictions, so the paired bootstrap here compares clipped
# vs. unclipped on identical resampled rows).
current_oof_clipped = np.clip(current_oof, 0.005, 0.995)
paired_gate("clip [0.005, 0.995] vs. unclipped (same predictions)",
            current_oof_clipped, current_oof)

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# CANDIDATE 3 precursor (Budd et al. 2023): does the disagreement already
# present across the 30 existing CNN members carry information the blend
# can use, before spending any GPU time on new asymmetric-loss ensembles?
# Per-row logit sd across the 30 members, plus its interaction with the
# pooled CNN logit, added to the existing 2-feature blend.
member_logits = np.array([to_logit(a) for a in cnn_arrays])  # shape (30, n_rows)
member_logit_sd = member_logits.std(axis=0, ddof=1)
cnn_logit_pooled = to_logit(cnn_pooled)

X_candidate3 = np.column_stack([
    cnn_logit_pooled,
    to_logit(baseline_pooled),
    member_logit_sd,
    member_logit_sd * cnn_logit_pooled,
])
candidate3_oof = oof_predict(X_candidate3, regularized=False)
result_c3 = paired_gate("candidate 3 precursor: + member-disagreement (sd) features",
                         candidate3_oof, current_oof)
print("pre-registered expectation: null (targets a calibration seam already "
      "measured at +0.0020 excess log loss -- see intro cell).")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# CANDIDATE 2 (Ithapu, Singh & Johnson 2015, Ch. 15): ridge/L2-regularized
# weighting across the 6 variant-level pooled logits + the baseline logit,
# instead of uniformly logit-pooling all 6 variants first. Regularization
# strength is tuned by nested CV (LogisticRegressionCV, inner_cv=5) INSIDE
# each outer fold's train rows only, per the review's explicit design
# constraint -- an unconstrained 30-member-level fit would just re-run the
# mechanism that already cost this project 0.916->0.889 real-test AUROC.
variant_pooled = {
    prefix: logit_mean([cnn_members[(prefix, s)] for s in repeat_seeds])
    for prefix in VARIANT_PREFIXES
}
X_candidate2 = np.column_stack(
    [to_logit(variant_pooled[prefix]) for prefix in VARIANT_PREFIXES]
    + [to_logit(baseline_pooled)]
)
candidate2_oof = oof_predict(X_candidate2, regularized=True)
result_c2 = paired_gate("candidate 2: ridge-weighted 6-variant + baseline (nested-CV C)",
                         candidate2_oof, current_oof)
print("pre-registered expectation: null (6 variants are highly correlated -- "
      "same architecture, different training knobs -- so a fitted weighting "
      "should converge close to uniform plus estimation noise).")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# CANDIDATE 1 remnant (Khachnaoui et al. 2023's fusion idea, scaled down to
# what's actually feasible here -- see intro cell for why the literal
# embedding-level bilinear fusion is structurally not viable): second-order
# interaction between the two existing blend scores, plus the classical
# baseline's own engineered features the 2-feature blend never sees
# directly. `inplane_family` one-hot adds real dimensionality (many rare
# categories) relative to n=1362 -- this is exactly the small-n/many-feature
# regime the nested-CV regularization is meant to guard against; flagged
# here rather than silently risking overfit.
family_dummies = pd.get_dummies(family, prefix="family").to_numpy(dtype=float)
print(f"inplane_family one-hot adds {family_dummies.shape[1]} columns "
      f"(some categories may be rare -- regularization handles this, not a manual collapse).")

cnn_logit_pooled = to_logit(cnn_pooled)
baseline_logit_pooled = to_logit(baseline_pooled)
X_candidate1 = np.column_stack([
    cnn_logit_pooled,
    baseline_logit_pooled,
    cnn_logit_pooled * baseline_logit_pooled,
    cnn_logit_pooled ** 2,
    abs_asym,
    striatal_ratio,
    family_dummies,
])
candidate1_oof = oof_predict(X_candidate1, regularized=True)
result_c1 = paired_gate("candidate 1: + interaction/quadratic/radiomics/family features",
                         candidate1_oof, current_oof)
print("pre-registered expectation: null (n=1362 rows for up to ~25 features "
      "gating a log-loss metric, in a project whose CV AUROC already drops "
      "0.027 on the real test set -- the textbook regime for a CV win that "
      "doesn't transfer).")

**What we're looking for:** whether today's three literature-sourced
candidates move the shipped recipe's honest row-wise CV log loss, using the
correct (paired-bootstrap) gate instead of notebook 22's unpaired-sd
statistic -- and, along the way, two free checks the Opus review asked for
before trusting 0.3617/0.8891 any further (the k=30-vs-k=150 extrapolation,
and the never-validated production probability clip).

**What we found:** *(paste: op04's shipped-recipe log loss/AUROC/ECE and
whether it matches notebook 22's 0.3617; op05's a_extrapolated_150 vs.
a_shipped difference; op06's clip delta and CI; op07/op08/op09's three
`paired_gate` verdicts -- score, delta, CI, and CLEARS/does-not-clear for
each)*

**Decision / next step:** *(per this notebook's pre-registered decision
rule -- whole CI negative AND |delta| > 0.003 -- adopt only what actually
clears; the pre-registered expectation for all three was null. If all three
came back null, per the Opus review's honest recommendation: the easy gains
are gone, calibration is exhausted (+0.0020 excess), and further gains can
only come from discrimination (AUROC), which none of today's three
candidates target. The only remaining lever with a discrimination mechanism
is roadmap item 6 (bounded architecture check, never run -- e.g. Wenzel et
al. 2019's 2D thick-slab CNN as a 7th, architecturally-diverse ensemble
member, already sourced in `RESOURCES.md`) -- worth 4-8h of work only if
[[project_dat_parkinson_strategic_roadmap]]'s open submission-budget
question resolves in favor of having a slot to spend it on. If something
DID clear the gate: it still needs the same rigor as every prior adoption
in this project -- wire into `src/submission.py`/`submission_src/main.py`
(TDD), rebuild submission assets, local Docker smoke test, platform smoke
test, before risking a real submission on it.)*